In [ ]:
import os
import pandas as pd
import random

# -------------------------
# CONFIG
# -------------------------
DATASET_PATH = r"D:\AIE\SEM 3\FAI project\originaldataset"
OUTPUT_CSV = r"D:\AIE\SEM 3\FAI project\final_voter_dataset.csv"

# -------------------------
# STEP 1: LOAD ORIGINAL DATASET
# -------------------------
print("Loading dataset...")

rows = []

for constituency in os.listdir(DATASET_PATH):
    const_path = os.path.join(DATASET_PATH, constituency)

    if not os.path.isdir(const_path):
        continue

    for voter_id in os.listdir(const_path):
        voter_path = os.path.join(const_path, voter_id)

        if not os.path.isdir(voter_path):
            continue

        for img in os.listdir(voter_path):
            if img.lower().endswith(('.jpg', '.png', '.jpeg')):
                img_path = os.path.join(voter_path, img)

                rows.append({
                    "voter_id": voter_id,
                    "registered_constituency": constituency,
                    "image_path": img_path
                })

df_legit = pd.DataFrame(rows)

print(f"Total images: {len(df_legit)}")

# -------------------------
# STEP 2: CREATE LEGIT RECORDS
# -------------------------
df_legit['claimed_voter_id'] = df_legit['voter_id']
df_legit['voting_constituency'] = df_legit['registered_constituency']
df_legit['fraud_type'] = "legitimate"
df_legit['is_suspicious'] = False

# -------------------------
# STEP 3: SIMULATE FRAUD
# -------------------------
print("Simulating fraud...")

fraud_rows = []

voters = df_legit['voter_id'].unique()
constituencies = df_legit['registered_constituency'].unique()

# Identity Theft
for _ in range(5):
    actual = random.choice(voters)
    claimed = random.choice([v for v in voters if v != actual])

    sample = df_legit[df_legit['voter_id'] == actual].iloc[0].copy()

    sample['claimed_voter_id'] = claimed
    sample['fraud_type'] = "identity_theft"
    sample['is_suspicious'] = True

    fraud_rows.append(sample)

# Cross Constituency
for _ in range(5):
    sample = df_legit.sample(1).iloc[0].copy()

    other = random.choice([c for c in constituencies if c != sample['registered_constituency']])

    sample['voting_constituency'] = other
    sample['fraud_type'] = "cross_constituency"
    sample['is_suspicious'] = True

    fraud_rows.append(sample)

# Double Voting
for vid in random.sample(list(voters), min(5, len(voters))):
    sample = df_legit[df_legit['voter_id'] == vid].iloc[0].copy()

    sample['fraud_type'] = "double_voting"
    sample['is_suspicious'] = True

    fraud_rows.append(sample)

df_fraud = pd.DataFrame(fraud_rows)

# -------------------------
# STEP 4: COMBINE
# -------------------------
df_final = pd.concat([df_legit, df_fraud], ignore_index=True)

# -------------------------
# STEP 5: CREATE IMAGE MAPPING (VERY IMPORTANT)
# -------------------------
print("Mapping images correctly...")

clean_rows = []

for _, row in df_final.iterrows():

    actual = row['voter_id']
    claimed = row['claimed_voter_id']

    voted_img = row['image_path']  # actual person image

    # 🔥 KEY FIX: get original image from claimed voter ONLY
    original_row = df_legit[df_legit['voter_id'] == claimed]

    if len(original_row) > 0:
        original_img = original_row.iloc[0]['image_path']
    else:
        original_img = "Not Found"

    clean_rows.append({
        "actual_voter_id": actual,
        "claimed_voter_id": claimed,
        "registered_constituency": row['registered_constituency'],
        "voting_constituency": row['voting_constituency'],
        "fraud_type": row['fraud_type'],
        "is_suspicious": row['is_suspicious'],
        "original_image_path": original_img,  # ✅ claimed voter
        "voted_image_path": voted_img         # ✅ actual voter
    })

# -------------------------
# STEP 6: FINAL DATAFRAME
# -------------------------
final_df = pd.DataFrame(clean_rows)

# Optional: sort
final_df = final_df.sort_values(by=["is_suspicious", "fraud_type"])

# -------------------------
# STEP 7: SAVE CSV
# -------------------------
final_df.to_csv(OUTPUT_CSV, index=False)

print("\n✅ FINAL CSV CREATED ")
print(f"Saved at: {OUTPUT_CSV}")

# -------------------------
# SUMMARY
# -------------------------
print("\n===== SUMMARY =====")
print(f"Total records: {len(final_df)}")
print(f"Fraud cases: {len(final_df[final_df['is_suspicious'] == True])}")

Loading dataset...
Total images: 1010
Simulating fraud...
Mapping images correctly...

✅ FINAL CSV CREATED (NO SWAP ISSUE)
Saved at: C:\Users\Thanusha\Desktop\project2\final_voter_dataset.csv

===== SUMMARY =====
Total records: 1025
Fraud cases: 15
